<a href="https://colab.research.google.com/github/SamShinwari/Advanced-AI-Bootcamp-2026/blob/main/Project_10_Do_ipynbcument_Similarity_using_Word2Vec_(News_Recommendation_System).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 10: Document Similarity using Word2Vec (News Recommendation System)

## Objective

In this project, we will build a **News Recommendation System** using the **Word2Vec** algorithm.

The system will:

- Convert each news article into a dense vector representation.
- Capture the semantic meaning of articles using Word2Vec embeddings.
- Measure similarity between news articles using Cosine Similarity.
- Recommend the most relevant news articles based on semantic similarity.

This concept is widely used in modern recommendation systems such as:

- 📰 Google News
- 📰 BBC News Recommendations
- ▶️ YouTube Video Recommendations
- 🎬 Netflix Movie Recommendations
- 🔍 Semantic Search Engines

---

# Project Workflow

```text
                 News Articles
                      │
                      ▼
              Text Cleaning
                      │
                      ▼
               Tokenization
                      │
                      ▼
             Train Word2Vec Model
                      │
                      ▼
             Generate Word Embeddings
                      │
                      ▼
        Compute Average Word Vectors
                      │
                      ▼
            Create Document Vectors
                      │
                      ▼
             Cosine Similarity
                      │
                      ▼
        Recommend Similar News Articles
```

---

# Steps Covered

1. Load the News Dataset
2. Explore the Dataset
3. Clean the Text
4. Tokenize News Articles
5. Train a Word2Vec Model
6. Generate Word Embeddings
7. Create Document Vectors
8. Compute Cosine Similarity
9. Recommend Similar News Articles
10. Visualize Similarity Results

---

# Expected Learning Outcomes

After completing this project, you will be able to:

- Understand document representation using Word2Vec.
- Convert variable-length documents into fixed-length vectors.
- Compute semantic similarity between documents.
- Build a simple content-based recommendation system.
- Apply cosine similarity for document retrieval.
- Understand the foundation of semantic search and recommendation engines.

---

# Step 1: Install Gensim

In [16]:
# Install Gensim library
!pip install gensim

In [17]:
import nltk

Download the English model:

In [18]:
# Download required resources (run once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

---

# Step 2: Import Libraries

In [19]:
import pandas as pd
import spacy
from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from google.colab import drive
from gensim.models import Word2Vec


In [20]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---

# Step 3: Load the English Model

In [22]:
nlp = spacy.load("en_core_web_sm")
# Download required resources (run once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

---

# Step 4: Load Dataset

In [23]:
file_path = '/content/drive/MyDrive/Colab Notebooks/bbc_text_cls.csv'
df = pd.read_csv(file_path)
df.head()

,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


In [24]:
df = df.dropna(subset=["text"])
df = df.drop_duplicates(subset=["text"])
df = df.reset_index(drop=True)

In [25]:
df = df.drop(columns=['labels'])
df.head()

,text
0,Ad sales boost Time Warner profit\n\nQuarterly...
1,Dollar gains on Greenspan speech\n\nThe dollar...
2,Yukos unit buyer faces loan claim\n\nThe owner...
3,High fuel prices hit BA's profits\n\nBritish A...
4,Pernod takeover talk lifts Domecq\n\nShares in...


In [26]:

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


True

In [27]:

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    words = text.split()
    # Remove stopwords and lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    # Join words back into a string
    return ' '.join(words)

# Apply the preprocessing function to the 'description' column to create 'clean_text'
df['clean_text'] = df['text'].apply(preprocess_text)

print(df[['text', 'clean_text']].head())

                                                text  \
0  Ad sales boost Time Warner profit\n\nQuarterly...   
1  Dollar gains on Greenspan speech\n\nThe dollar...   
2  Yukos unit buyer faces loan claim\n\nThe owner...   
3  High fuel prices hit BA's profits\n\nBritish A...   
4  Pernod takeover talk lifts Domecq\n\nShares in...   

                                          clean_text  
0  ad sale boost time warner profit quarterly pro...  
1  dollar gain greenspan speech dollar hit highes...  
2  yukos unit buyer face loan claim owner embattl...  
3  high fuel price hit ba profit british airway b...  
4  pernod takeover talk lift domecq share uk drin...  


---

# Step 4: Tokenize the Text

Word2Vec expects a list of tokenized sentences. We will use the `clean_text` column and split each string into a list of words.

In [28]:
sentences = df["clean_text"].apply(str.split)

print(sentences.head())

0    [ad, sale, boost, time, warner, profit, quarte...
1    [dollar, gain, greenspan, speech, dollar, hit,...
2    [yukos, unit, buyer, face, loan, claim, owner,...
3    [high, fuel, price, hit, ba, profit, british, ...
4    [pernod, takeover, talk, lift, domecq, share, ...
Name: clean_text, dtype: object


---

# Step 4: Train Word2Vec

In [29]:
model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0
)

---

# Step 5: Convert One Document into a Vector

Each document is represented by the **average of all its word vectors**.

In [30]:
def document_vector(words, model):

    words = [word for word in words if word in model.wv]

    if len(words) == 0:
        return np.zeros(model.vector_size)

    return np.mean(model.wv[words], axis=0)

---

# Step 6: Create Document Vectors

In [31]:
document_vectors = []

for sentence in sentences:

    vector = document_vector(sentence, model)

    document_vectors.append(vector)

document_vectors = np.array(document_vectors)

print(document_vectors.shape)

(2127, 100)


Example Output

```text
(2225,100)
```

Meaning

* 2225 news articles
* Each represented by a 100-dimensional vector

---

# Step 7: Compute Cosine Similarity

In [32]:
similarity_matrix = cosine_similarity(document_vectors)

print(similarity_matrix.shape)

(2127, 2127)


Output

```text
(2225,2225)
```

Each row compares one article with every other article.

---

# Step 8: Find Similar News

Let's choose article **0**.

In [33]:
article_index = 0

scores = similarity_matrix[article_index]

---

# Step 9: Sort Similar Articles

In [34]:
similar_articles = np.argsort(scores)[::-1]

print(similar_articles[:10])

[  0 139 123  49  23 318 457 108  94 142]


Example

```text
0
25
102
18
201
90
```

---

# Step 10: Display Recommendations

In [35]:
print("Original News\n")

print(df.iloc[article_index]["text"])

print("\nRecommended News\n")

for idx in similar_articles[1:6]:

    print(df.iloc[idx]["text"])

Original News

Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.

Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try

---

# Step 11: Search by Query

Instead of selecting an article, search by text.

In [36]:
query = "Ukraine war"

query = query.lower().split()

query_vector = document_vector(query, model)

---

# Step 12: Compute Similarity

In [37]:
scores = cosine_similarity(
    [query_vector],
    document_vectors
)

---

# Step 13: Top Results

In [38]:
top = np.argsort(scores[0])[::-1][:5]

for i in top:

    print(df.iloc[i]["text"])

Ethnic producers 'face barriers'

Minority ethnic led (Mel) production companies face barriers in succeeding in the film and television industries, research has suggested.

The study, commissioned by Pact and the UK Film Council, included interviews with industry experts and individuals. They indicated that career progression and a lack of role models are among the main problems within such companies. The research indicated that about 10% of independent production companies in the UK are minority ethnic led.

A minority ethnic led company is defined as one in which the majority of decision-making power rests with an individual or individuals from a minority ethnic group. The report also explored the problems faced by such companies when attempting to compete within the film and TV industries. It said they are often smaller than other companies and lack the resources, so are often squeezed out of the market by bigger firms. The research recommended that minority ethnic led companies cou

---

# Step 14: Build a Recommendation Function

In [39]:
def recommend_news(query, model, document_vectors, df, top_n=5):

    query = query.lower().split()

    query_vector = document_vector(query, model)

    similarity = cosine_similarity(
        [query_vector],
        document_vectors
    )[0]

    top = np.argsort(similarity)[::-1][:top_n]

    for index in top:

        print(df.iloc[index]["text"])

---

# Step 15: Test

In [40]:
recommend_news(
    "Ukraine war",
    model,
    document_vectors,
    df
)

Ethnic producers 'face barriers'

Minority ethnic led (Mel) production companies face barriers in succeeding in the film and television industries, research has suggested.

The study, commissioned by Pact and the UK Film Council, included interviews with industry experts and individuals. They indicated that career progression and a lack of role models are among the main problems within such companies. The research indicated that about 10% of independent production companies in the UK are minority ethnic led.

A minority ethnic led company is defined as one in which the majority of decision-making power rests with an individual or individuals from a minority ethnic group. The report also explored the problems faced by such companies when attempting to compete within the film and TV industries. It said they are often smaller than other companies and lack the resources, so are often squeezed out of the market by bigger firms. The research recommended that minority ethnic led companies cou

Try another query:

In [41]:
recommend_news(
    "Covid vaccine",
    model,
    document_vectors,
    df
)

France Telecom gets Orange boost

Strong growth in subscriptions to mobile phone network Orange has helped boost profits at owner France Telecom.

Orange added more than five million new customers in 2004, leading to a 10% increase in its revenues. Increased take-up of broadband telecoms services also boosted France Telecom's profits, which showed a 5.5% rise to 18.3bn euros ($23.4bn; £12.5bn). France Telecom is to spend 578m euros on buying out minority shareholders in data services provider Equant.

France Telecom, one of the world's largest telecoms and internet service providers, saw its full-year sales rise 2.2% to 47.2bn euros in 2004.

Orange enjoyed strong growth outside France and the United Kingdom - its core markets - swelling its subscriber base to 5.4 million. France Telecom's broadband customers also increased, rising to 5.1 million across Europe by the end of the year. The firm said it had met its main strategic objectives of growing its individual businesses and further

Or

In [42]:
recommend_news(
    "Manchester football",
    model,
    document_vectors,
    df
)

'Landmark movies' of 2004 hailed

US film professionals have declared Fahrenheit 9/11 and The Passion of the Christ as two of the most significant cultural milestones of 2004.

The American Film Institute (AFI) hailed Mel Gibson's biblical epic and Michael Moore's political documentary as inspiring national debate. It claimed both film-makers "tossed Hollywood convention out the window". The Institute also cited the death of actor Marlon Brando and the changing landscape of TV news in the US. In referring to Marlon Brando's death on 1 July at the age of 80, the 13-strong AFI jury concluded "the art of screen acting has two chapters - 'Before Brando' and 'After Brando'.

It credited the screen legend's "raw hypnotic energy" and his ability to create characters like Stanley Kowalski and Terry Malloy "that will live forever in the annals of film history". The list also acknowledges key influences and trends in the world of film and broadcasting. Among current trends, it highlighted the fi

---

# Step 16: Save Document Vectors

In [43]:
np.save(
    "document_vectors.npy",
    document_vectors
)

---

# Complete Code

---

# Advantages

* Captures semantic meaning
* Better than keyword matching
* Learns relationships between words
* Fast recommendation system
* Easy to scale

---

# Limitations

* Averaging word vectors ignores word order.
* Documents with different lengths are treated similarly.
* Less effective than modern transformer embeddings (e.g., Sentence-BERT).

---

# Real-World Applications

* News recommendation
* Similar article detection
* Semantic document search
* Duplicate news detection
* Content recommendation
* Search engines